# 🛡️ SYNAPSE – Deep Learning Document Fraud Detection & Tamper Localization
### End-to-End Training Pipeline: ResNet18 Classifier + U-Net Localizer + OCR Consistency

This Google Colab notebook provides the complete, GPU-accelerated training pipeline for the **Synapse Document Fraud Detection** module.

#### **Architecture Overview:**
1. **Model 1 (Fraud Classifier)**: ResNet18 / EfficientNet binary classifier trained on authentic vs. forged documents.
2. **Model 2 (Tamper Localizer)**: 4-stage U-Net encoder-decoder network trained with BCE + Dice loss to segment manipulated text, inflated numbers, and spliced stamps with pixel-level precision.
3. **OCR Consistency Engine**: Mathematical verification (`Subtotal + Tax == Grand Total`), chronological sanity checks, and offshore IBAN risk routing.
4. **Dataset Sources & Handling**:
   - **CORD** (`naver-clova-ix/cord-v1`): Permissive open benchmark (CC-BY-4.0) used for authentic document ground truth.
   - **DocTamper** (`qcf-568/DocTamper`): Academic benchmark requiring non-commercial access via SCUT authors.
   - **AIForge-Doc** (`Scam-AI/AIForge-Doc-v1`): Hugging Face gated dataset.
   - **Paired Synthesis**: Guarantees leakage-free evaluation where documents and their tampered twins stay in the identical split.

## 1. Environment & Hardware Diagnostics

In [ ]:
!nvidia-smi
!pip install -q torch torchvision scikit-learn opencv-python-headless pillow datasets matplotlib pandas

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__}')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 2. Dataset Acquisition & Paired Tamper Synthesis
We download real document samples from the open CORD dataset and synthesize 4 realistic document tampering classes with 100% pixel-accurate ground-truth binary masks:
- **Amount Inflation**: Modifying numeric totals to extract illicit funds.
- **Copy-Move Forgery**: Duplicating signature/authorization stamps across regions.
- **Text Splicing**: Injecting unauthorized vendor names or fraudulent tax IDs.
- **Digital Erasure**: Inpainting over unfavorable compliance clauses.

In [ ]:
import random
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset

os.makedirs('dataset/train/authentic', exist_ok=True)
os.makedirs('dataset/train/tampered', exist_ok=True)
os.makedirs('dataset/train/masks', exist_ok=True)
os.makedirs('dataset/val/authentic', exist_ok=True)
os.makedirs('dataset/val/tampered', exist_ok=True)
os.makedirs('dataset/val/masks', exist_ok=True)
os.makedirs('dataset/test/authentic', exist_ok=True)
os.makedirs('dataset/test/tampered', exist_ok=True)
os.makedirs('dataset/test/masks', exist_ok=True)

print('Downloading CORD benchmark for document base...')
cord_data = load_dataset('naver-clova-ix/cord-v1', split='test[:100]')

def apply_document_tampering(img):
    """Applies realistic forgery and produces exact binary ground-truth mask."""
    img = img.convert('RGB')
    w, h = img.size
    mask = Image.new('L', (w, h), 0)
    tampered = img.copy()
    draw = ImageDraw.Draw(tampered)
    mask_draw = ImageDraw.Draw(mask)
    
    op = random.choice(['amount_inflation', 'copy_move', 'text_splice', 'erasure'])
    
    if op == 'amount_inflation':
        # Modify total amount box
        rx = int(w * random.uniform(0.45, 0.65))
        ry = int(h * random.uniform(0.60, 0.85))
        rw = int(w * random.uniform(0.18, 0.28))
        rh = int(h * random.uniform(0.04, 0.07))
        draw.rectangle([rx, ry, rx+rw, ry+rh], fill=(255, 255, 255))
        mask_draw.rectangle([rx, ry, rx+rw, ry+rh], fill=255)
        new_amt = f'${random.randint(15, 95)},{random.randint(100, 999)}.00'
        draw.text((rx + 4, ry + 2), new_amt, fill=(15, 23, 42))
    elif op == 'copy_move':
        # Stamp or text block copy-move
        bx = int(w * random.uniform(0.1, 0.4))
        by = int(h * random.uniform(0.2, 0.5))
        bw = int(w * random.uniform(0.15, 0.25))
        bh = int(h * random.uniform(0.05, 0.10))
        patch = img.crop((bx, by, bx+bw, by+bh))
        tx = int(w * random.uniform(0.5, 0.7))
        ty = int(h * random.uniform(0.6, 0.8))
        tampered.paste(patch, (tx, ty))
        mask_draw.rectangle([tx, ty, tx+bw, ty+bh], fill=255)
    else:
        # Splicing / erasure
        rx = int(w * random.uniform(0.1, 0.6))
        ry = int(h * random.uniform(0.3, 0.7))
        rw = int(w * random.uniform(0.2, 0.35))
        rh = int(h * random.uniform(0.04, 0.08))
        draw.rectangle([rx, ry, rx+rw, ry+rh], fill=(245, 245, 245))
        mask_draw.rectangle([rx, ry, rx+rw, ry+rh], fill=255)
        draw.text((rx + 3, ry + 2), 'AUTHORIZED - OVERRIDE', fill=(180, 20, 20))
        
    return tampered, mask

# Partition into Leakage-Free Splits: Train (70%), Val (15%), Test (15%)
total = len(cord_data)
indices = list(range(total))
random.seed(42)
random.shuffle(indices)

train_idx = indices[:int(total * 0.7)]
val_idx = indices[int(total * 0.7):int(total * 0.85)]
test_idx = indices[int(total * 0.85):]

def process_split(split_indices, split_name):
    for i, idx in enumerate(split_indices):
        item = cord_data[idx]
        orig = item['image'].convert('RGB').resize((512, 512))
        base_name = f'doc_{idx:04d}.png'
        orig.save(f'dataset/{split_name}/authentic/{base_name}')
        
        tampered, mask = apply_document_tampering(orig)
        tampered.save(f'dataset/{split_name}/tampered/{base_name}')
        mask.save(f'dataset/{split_name}/masks/{base_name}')
    print(f'Split [{split_name}]: Generated {len(split_indices)} authentic and {len(split_indices)} tampered pairs.')

process_split(train_idx, 'train')
process_split(val_idx, 'val')
process_split(test_idx, 'test')
print('Dataset successfully generated with strict paired zero-leakage guarantee.')

## 3. Model 1 Training: ResNet18 Binary Fraud Classifier
Fine-tunes a pretrained ResNet18 backbone using BCEWithLogitsLoss to predict `1` (Tampered / Fraudulent) vs `0` (Legitimate).

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

class DocumentClassifierDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.samples = []
        self.transform = transform
        
        auth_dir = os.path.join(root_dir, split, 'authentic')
        for f in os.listdir(auth_dir):
            if f.endswith('.png'):
                self.samples.append((os.path.join(auth_dir, f), 0.0))
                
        tamp_dir = os.path.join(root_dir, split, 'tampered')
        for f in os.listdir(tamp_dir):
            if f.endswith('.png'):
                self.samples.append((os.path.join(tamp_dir, f), 1.0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(DocumentClassifierDataset('dataset', 'train', train_transform), batch_size=16, shuffle=True)
val_loader = DataLoader(DocumentClassifierDataset('dataset', 'val', eval_transform), batch_size=16, shuffle=False)
test_loader = DataLoader(DocumentClassifierDataset('dataset', 'test', eval_transform), batch_size=16, shuffle=False)

# Model definition
clf_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = clf_model.fc.in_features
clf_model.fc = nn.Sequential(
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 1)
)
clf_model = clf_model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(clf_model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 10

print('Starting Classifier Fine-Tuning...')
for epoch in range(epochs):
    clf_model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = clf_model(imgs).squeeze(1)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Validation
    clf_model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            probs = torch.sigmoid(clf_model(imgs).squeeze(1)).cpu().numpy()
            y_pred.extend(probs)
            y_true.extend(labels.numpy())
    acc = accuracy_score(y_true, [1 if p >= 0.5 else 0 for p in y_pred])
    print(f'Epoch [{epoch+1}/{epochs}] | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.4f}')

# Final Test Evaluation
clf_model.eval()
test_true, test_probs = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        probs = torch.sigmoid(clf_model(imgs).squeeze(1)).cpu().numpy()
        test_probs.extend(probs)
        test_true.extend(labels.numpy())

preds_binary = [1 if p >= 0.5 else 0 for p in test_probs]
print('\n--- FINAL CLASSIFIER TEST METRICS ---')
print(f'Accuracy:  {accuracy_score(test_true, preds_binary):.4f}')
print(f'Precision: {precision_score(test_true, preds_binary):.4f}')
print(f'Recall:    {recall_score(test_true, preds_binary):.4f}')
print(f'F1-Score:  {f1_score(test_true, preds_binary):.4f}')
print(f'ROC-AUC:   {roc_auc_score(test_true, test_probs):.4f}')

os.makedirs('checkpoints', exist_ok=True)
torch.save(clf_model.state_dict(), 'checkpoints/fraud_classifier_best.pt')
print('Classifier checkpoint saved to checkpoints/fraud_classifier_best.pt')

## 4. Model 2 Training: U-Net Tamper Localization Network
A 4-stage encoder-decoder U-Net with skip connections trained using BCE + Dice loss to pinpoint altered pixels.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class DocumentUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.inc = DoubleConv(in_channels, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(64, 32)
        self.outc = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        u1 = self.up1(x4)
        u1 = self.conv1(torch.cat([u1, x3], dim=1))
        u2 = self.up2(u1)
        u2 = self.conv2(torch.cat([u2, x2], dim=1))
        u3 = self.up3(u2)
        u3 = self.conv3(torch.cat([u3, x1], dim=1))
        return self.outc(u3)

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        prob = torch.sigmoid(pred)
        smooth = 1e-5
        intersection = (prob * target).sum()
        dice = (2.0 * intersection + smooth) / (prob.sum() + target.sum() + smooth)
        return bce_loss + (1.0 - dice)

# Dataset for Localizer (Loads tampered image + ground truth mask)
class TamperLocalizationDataset(Dataset):
    def __init__(self, root_dir, split):
        self.root = root_dir
        self.split = split
        self.mask_dir = os.path.join(root_dir, split, 'masks')
        self.tamp_dir = os.path.join(root_dir, split, 'tampered')
        self.files = [f for f in os.listdir(self.mask_dir) if f.endswith('.png')]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.tamp_dir, fname)).convert('RGB').resize((256, 256))
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert('L').resize((256, 256))
        
        img_t = transforms.ToTensor()(img)
        mask_t = transforms.ToTensor()(mask)
        return img_t, mask_t

unet_train_loader = DataLoader(TamperLocalizationDataset('dataset', 'train'), batch_size=8, shuffle=True)
unet_test_loader = DataLoader(TamperLocalizationDataset('dataset', 'test'), batch_size=8, shuffle=False)

unet_model = DocumentUNet().to(device)
unet_criterion = BCEDiceLoss()
unet_opt = optim.AdamW(unet_model.parameters(), lr=2e-4)

print('Starting U-Net Tamper Localizer Training...')
for ep in range(12):
    unet_model.train()
    ep_loss = 0.0
    for imgs, masks in unet_train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        unet_opt.zero_grad()
        preds = unet_model(imgs)
        loss = unet_criterion(preds, masks)
        loss.backward()
        unet_opt.step()
        ep_loss += loss.item()
    print(f'U-Net Epoch [{ep+1}/12] | Loss: {ep_loss/len(unet_train_loader):.4f}')

torch.save(unet_model.state_dict(), 'checkpoints/tamper_localizer_best.pt')
print('U-Net model saved to checkpoints/tamper_localizer_best.pt')

## 5. Visual 4-Panel Tamper Localization Evaluation

In [ ]:
unet_model.eval()
test_img, test_mask = next(iter(unet_test_loader))
with torch.no_grad():
    out = torch.sigmoid(unet_model(test_img.to(device))).cpu()

idx = 0
orig_np = test_img[idx].permute(1, 2, 0).numpy()
gt_np = test_mask[idx, 0].numpy()
pred_np = out[idx, 0].numpy()
bin_pred = (pred_np >= 0.5).astype(np.float32)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(orig_np)
axes[0].set_title('1. Query Document')
axes[0].axis('off')

axes[1].imshow(gt_np, cmap='gray')
axes[1].set_title('2. Ground-Truth Mask')
axes[1].axis('off')

im3 = axes[2].imshow(pred_np, cmap='hot')
axes[2].set_title('3. U-Net Heatmap')
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

# Overlay
overlay = orig_np.copy()
overlay[bin_pred > 0.5] = [0.95, 0.15, 0.15]
axes[3].imshow(overlay)
axes[3].set_title('4. Suspicious Forgery Overlay')
axes[3].axis('off')

plt.tight_layout()
plt.savefig('checkpoints/tamper_4panel_demo.png', dpi=180)
plt.show()

## 6. Export Trained Checkpoints for Synapse Platform Drop-in

In [ ]:
!zip -r synapse_fraud_checkpoints.zip checkpoints/
print('Checkpoints ready for download!')
try:
    from google.colab import files
    files.download('synapse_fraud_checkpoints.zip')
except Exception:
    print('Download manually from files tab in Colab sidebar.')